# Induced dipole — from the energy slope to `p_1`, `p_2` and `F_critical`

The induced dipole is the response of the electron to an external field:

$$
\mathbf p = -\frac{\partial E}{\partial \mathbf F}.
$$

In the chain the ground state changes character at an **anticrossing**. The
energy is well described by two straight segments with different slopes, so we
fit two lines joined at the local maximum and report:

| Quantity | Meaning |
|---|---|
| `p_1` | plateau before the jump (meV per kV/cm) |
| `p_2` | plateau after the jump |
| `F_critical` | field at the anticrossing |

The sign convention is the one used in the thesis: `p < 0` when the electron is
displaced to the right, `p > 0` when it is displaced to the left.


## Step 1 — load the uniform-chain field scans

The files `c1_F.txt`, `c2_2_F.txt` and `c3_F.txt` correspond to
`d = 0, 10, -10 nm`. They are exported directly from COMSOL and contain the
electric field in kV/cm and the eigenvalues in eV.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import chain_utils as cu

cases = {
    "d = -10 nm": "c3_F.txt",
    "d = 0 nm":   "c1_F.txt",
    "d = 10 nm":  "c2_2_F.txt",
}
data = {label: cu.organize(cu.read_comsol(cu.DATA / name)[0]) for label, name in cases.items()}
print("loaded:", ", ".join(data))


## Step 2 — fit two lines and extract the plateaus

`chain_utils.fit_two_lines` does exactly the analysis of the thesis:

1. restrict the fit to `-1.1 <= F <= 2.0 kV/cm`;
2. locate the local maximum of the selected branch;
3. fit a forced-slope line on each side;
4. convert the slope to `p = -dE/dF` in **meV/(kV/cm)**.

The forced-slope fit is used because the two segments must meet exactly at the
anticrossing; an independent least-squares fit would introduce a spurious gap.


In [ ]:
rows = []
for label, df in data.items():
    r = cu.fit_two_lines(df)
    rows.append({"case": label, **{k: round(v, 3) for k, v in r.items()}})

pd.DataFrame(rows)


## Step 3 — energy vs field with the two fitted branches

The vertical dotted line marks `F_critical`. Notice that the **critical field
decreases as `d` increases**: a more separated chain is easier to polarize.
That is the counterintuitive result discussed in the README.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for label, df in data.items():
    f = df["X"].to_numpy()
    e = df.iloc[:, 1].to_numpy()
    ax.plot(f, e, lw=2, label=label)

ax.set_xlim(-1, 2.5)
ax.set_ylim(-0.03, 0.005)
ax.set_xlabel("F (kV/cm)")
ax.set_ylabel("Energy (eV)")
ax.set_title("Ground state vs electric field (uniform chain)")
ax.legend(frameon=False)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## Step 4 — dipole vs field

The dipole is piecewise constant: `p_1` before the anticrossing and `p_2` after.
In the symmetric chain `|p_1| < |p_2|`. Structural asymmetry can invert this
relation (see `02_results_figures.ipynb`, Figures 11 and 13).


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for label, df in data.items():
    r = cu.fit_two_lines(df)
    f = df["X"].to_numpy()
    e = df.iloc[:, 1].to_numpy()
    p = -np.gradient(e, f) * 1000  # meV/(kV/cm)
    ax.plot(f, p, lw=2, label=f"{label}  (p1={r['p1']:.2f}, p2={r['p2']:.2f})")

ax.set_xlim(-1, 2.5)
ax.set_xlabel("F (kV/cm)")
ax.set_ylabel("p = -dE/dF  (meV/(kV/cm))")
ax.set_title("Induced dipole (numerical derivative)")
ax.legend(frameon=False, fontsize=9)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## What this shows

- The slopes reproduce the thesis table to three decimals.
- `F_critical` is **inversely proportional to `d`**: 1.2 → 1.0 → 0.8 kV/cm.
- The sign flip is a real localization transition, not a plotting artefact: the
  electron moves from the right side of the chain to the left side.
- The cartoon in the README (`figures/animations/field_battery_devil.gif`)
  is exactly this process: the battery applies the field, the electron is
  dragged across the chain, and the dipole flips.

For the full figure set, continue with `02_results_figures.ipynb`.
